# DSC LOGIKA UI 2026 — Prediksi Default Kredit (notebook final, kandidat A)

**Ringkasan.** Prediksi peluang default (DPD 90 dalam 12 bulan) untuk 30.000 aplikasi test, metrik KS.
Pipeline: 528 fitur rekayasa dari 8 tabel + fitur **rantai bayar bulanan level event** (temuan eksplorasi) + fitur
**model level order e-commerce** (nested per fold) → dua model LightGBM (GBDT tuned dan DART), masing-masing rata-rata
3 seed × 5 fold → blend rank 50:50.

| komponen | mean-fold AUC (OOF, 5-fold) |
|---|---|
| GBDT tuned (E205) | 0,84517 |
| DART 5000 ronde (E207) | 0,84536 |
| **blend rank 50:50 (submission)** | **0,84545** (KS OOF 0,5242; public 0,56428) |

**Reproduksibilitas.** Semua fitur dihitung dari data mentah di notebook ini (tanpa cache), seed tetap, LightGBM
deterministik dengan jumlah thread tetap, file ditulis dengan pembulatan 6 desimal dan akhir baris LF. Output harus
identik dengan submission terpilih: **MD5 `afc487f68e4603a38209f2d4a472312b`**. Semua model CPU (tanpa GPU).
Perkiraan waktu Run All: ±3–3,5 jam di CPU 4 thread (DART paling lama, ±2,3 jam). Ganti `DATA_DIR` bila perlu.

## 1. Setup
Versi pustaka yang dipakai saat submission dibuat dicetak di bawah; pakai versi yang sama untuk hasil identik.

In [ ]:
import gc
import glob
import hashlib
import os
import platform
import sys
import time
import warnings
from functools import partial
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.stats import ks_2samp, rankdata
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
SEED = 42
N_FOLDS = 5
N_THREADS = 4
RUN_TUNING = False  # parameter hasil Optuna (Fase 5) di-hardcode di bagian 7; tuning tidak diulang

_hits = glob.glob("/kaggle/input/**/train_applications.csv", recursive=True)
DATA_DIR = Path(os.path.dirname(_hits[0])) if _hits else Path(os.environ.get("DSC_DATA_DIR", "Dataset"))
OUT_DIR = Path("/kaggle/working") if os.path.isdir("/kaggle/working") else Path(".")
OUT_PATH = OUT_DIR / "submission.csv"
EXPECTED = {"python": "3.12", "lightgbm": "4.7.0", "numpy": "2.3.3", "pandas": "2.3.2", "scikit-learn": "1.8.0",
            "scipy": "1.16.3"}
print("python", platform.python_version(), "| lightgbm", lgb.__version__, "| numpy", np.__version__,
      "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__, "| scipy", scipy.__version__)
print("versi saat submission dibuat:", EXPECTED)
print("DATA_DIR =", DATA_DIR.resolve(), "| OUT_PATH =", OUT_PATH.resolve())
T0 = time.time()

## 2. Data

Delapan tabel panitia (folder `DATA_DIR`). Kunci join `customer_id` (1 customer = 1 aplikasi train/test).
Semua fitur tabel pendukung dihitung relatif terhadap `decision_datetime` aplikasi sekarang (cutoff); seluruh
timestamp tabel pendukung sudah < cutoff (diverifikasi saat EDA). Tanggal di-parse dan kolom kategori dibaca
sebagai `category` supaya hemat memori.

In [ ]:
_DATES = {
    "train_applications": ["application_datetime", "decision_datetime"],
    "test_applications": ["application_datetime", "decision_datetime"],
    "previous_applications": ["application_datetime", "decision_datetime"],
    "repayment_history": ["due_date", "last_payment_datetime"],
    "bureau_observations": ["report_datetime", "available_datetime"],
    "account_behavior": ["month"],
    "ecommerce_transactions": ["transaction_datetime"],
}

_CATS = {
    "previous_applications": ["decision"],
    "bureau_observations": ["bureau_provider"],
    "ecommerce_transactions": ["product_category", "payment_method", "order_status"],
}


def load(name, usecols=None, data_dir=DATA_DIR):
    """Baca satu tabel dengan parse tanggal dan dtype kategori.

    Kolom ID string (customer_id, order_id, ...) dibiarkan object karena dipakai untuk join.
    """
    dates = [c for c in _DATES.get(name, []) if usecols is None or c in usecols]
    dtype = {c: "category" for c in _CATS.get(name, []) if usecols is None or c in usecols}
    return pd.read_csv(data_dir / f"{name}.csv", usecols=usecols, parse_dates=dates, dtype=dtype)


def load_apps(data_dir=DATA_DIR):
    """Train + test digabung (test punya target NaN), diurutkan train dulu lalu test."""
    tr = load("train_applications", data_dir=data_dir)
    te = load("test_applications", data_dir=data_dir)
    tr["is_test"] = 0
    te["is_test"] = 1
    return pd.concat([tr, te], ignore_index=True)


def cutoffs(apps):
    """Peta customer_id -> application_id dan cutoff (= decision_datetime aplikasi saat ini)."""
    return apps[["customer_id", "application_id", "decision_datetime"]].rename(
        columns={"decision_datetime": "cutoff"}
    )


apps = load_apps()
m_tr = apps["is_test"].eq(0).values
print(apps.shape, "| target rate train", round(apps.loc[m_tr, "target"].mean(), 4))

## 3. Ringkasan EDA (temuan yang membentuk pipeline)

- Split train/test acak (bukan temporal), customer tidak overlap → StratifiedKFold 5 fold valid sebagai CV.
- Target 13,58%; 30% customer tanpa riwayat (target lebih rendah, 11,7%) → missing dibiarkan NaN untuk GBDT.
- Data dibangkitkan dengan **risiko laten customer yang stabil**: label default antarpinjaman lama sangat berulang
  (ICC 0,40); target sekarang berperilaku seperti "satu pinjaman lagi".
- **Pembayaran cicilan hanya terjadi tepat di ulang bulan pencairan** (rantai Markov bulanan; FIFO; 0–3 cicilan per
  event) → representasi level event (bagian 5.9) jauh lebih informatif daripada agregat baris cicilan.
- E-commerce dipotong tepat 24 jam sebelum keputusan; order defaulter lebih rapat menjelang keputusan ("ramp") →
  fitur jendela pendek dan model level order (bagian 6).
- `annual_interest_rate` = 5 tier dengan target rate monoton 5,5% → 50,3% (grade risiko lender).

In [ ]:
tr_apps = apps[m_tr]
print("target per tier bunga:", tr_apps.groupby("annual_interest_rate")["target"].mean().round(3).to_dict())
print("aplikasi per bulan keputusan:", tr_apps.decision_datetime.dt.to_period("M").value_counts().sort_index().to_dict())

## 4. Kontrak fitur

Setiap builder `build_<grup>(apps)` mengembalikan DataFrame ber-index `application_id`, kolom ber-prefix dan terurut.
`cached` di pipeline eksperimen menyimpan hasil ke parquet; di notebook ini hasil disimpan di memori (nilainya sama).

In [ ]:
def finalize(df, prefix):
    """Tambah prefix (jika belum), urutkan kolom, cast float64 -> float32 untuk hemat memori."""
    df = df.rename(columns={c: c if c.startswith(prefix) else f"{prefix}{c}" for c in df.columns})
    df = df[sorted(df.columns)]
    f64 = df.select_dtypes("float64").columns
    df[f64] = df[f64].astype("float32")
    return df


def by_customer(agg, apps, prefix):
    """Peta agregat per customer_id -> per application_id (customer tanpa data -> NaN)."""
    out = apps[["application_id", "customer_id"]].merge(
        agg, left_on="customer_id", right_index=True, how="left"
    ).set_index("application_id").drop(columns="customer_id")
    return finalize(out, prefix)


_CACHE = {}


def cached(name, builder, apps, rebuild=False):
    """Pengganti cache parquet: hasil grup fitur disimpan di memori, di-reindex ke urutan apps."""
    if name in _CACHE and not rebuild:
        return _CACHE[name]
    df = builder(apps)
    df = df.reindex(apps["application_id"])
    df.index.name = "application_id"
    _CACHE[name] = df
    return df


def days_between(later, earlier):
    return (later - earlier).dt.total_seconds() / 86400

## 5. Rekayasa fitur
Kode di bawah identik dengan modul `src/features/` pipeline eksperimen (hanya import yang dihapus).

### 5.1 Aplikasi sekarang dan profil (`app_`, `prof_`)

Fitur aplikasi saat ini (app_) dan profil customer (prof_).

In [ ]:
APP_CATS = {"loan_purpose": ["education", "home_improvement", "household", "medical",
                             "working_capital"]}
PROF_CATS = {
    "region": ["Region_A", "Region_B", "Region_C", "Region_D", "Region_E"],
    "employment_type": ["salaried", "self_employed", "variable_income"],
    "housing_status": ["family", "owned", "rented"],
}


def _to_cat(df, cats):
    for c, levels in cats.items():
        df[c] = pd.Categorical(df[c], categories=levels)
    return df


def build_app_raw(apps):
    """E01: kolom mentah aplikasi. term_months dibuang (konstan 12)."""
    cols = ["loan_amount", "annual_interest_rate", "proposed_monthly_installment", "loan_purpose"]
    df = apps.set_index("application_id")[cols].copy()
    return finalize(_to_cat(df, APP_CATS), "app_")


def build_prof_raw(apps):
    """E01: kolom mentah profil customer."""
    prof = load("customer_profiles")
    df = apps[["application_id", "customer_id"]].merge(prof, on="customer_id", how="left")
    df = df.set_index("application_id").drop(columns="customer_id")
    return finalize(_to_cat(df, PROF_CATS), "prof_")


def build_app_ratios(apps):
    """E02: keterjangkauan, stabilitas kerja, dan waktu proses aplikasi."""
    prof = load("customer_profiles")
    a = apps.merge(prof, on="customer_id", how="left").set_index("application_id")
    inc = a["declared_monthly_income"].replace(0, np.nan)
    out = pd.DataFrame(index=a.index)
    out["pti"] = a["proposed_monthly_installment"] / inc
    out["lti"] = a["loan_amount"] / inc
    out["income_per_member"] = a["declared_monthly_income"] / (a["dependents"] + 1)
    out["pti_per_member"] = a["proposed_monthly_installment"] / out["income_per_member"]
    out["residual_income"] = a["declared_monthly_income"] - a["proposed_monthly_installment"]
    out["tenure_share_of_age"] = a["employment_tenure_months"] / (a["age"] * 12)
    out["age_started_job"] = a["age"] - a["employment_tenure_months"] / 12
    out["process_hours"] = days_between(a["decision_datetime"], a["application_datetime"]) * 24
    out["app_hour"] = a["application_datetime"].dt.hour
    out["app_dayofweek"] = a["application_datetime"].dt.dayofweek
    out["app_month"] = a["application_datetime"].dt.month
    out["dec_hour"] = a["decision_datetime"].dt.hour
    out["total_interest"] = a["proposed_monthly_installment"] * 12 - a["loan_amount"]
    return finalize(out, "app_")

### 5.2 Agregat naif semua tabel pendukung

E03: agregat naif (count/mean/max/min/std) untuk semua tabel pendukung.

Baseline B1. Fitur yang lebih cermat per tabel (E10 dst.) ditulis di modul per tabel.
Semua recency dihitung terhadap cutoff = decision_datetime aplikasi saat ini.

In [ ]:
STATS = ["mean", "max", "min", "std"]


def _num_agg(df, cols, stats=STATS):
    agg = df.groupby("customer_id")[cols].agg(stats)
    agg.columns = [f"{c}_{s}" for c, s in agg.columns]
    return agg


def _with_cutoff(df, apps):
    return df.merge(cutoffs(apps)[["customer_id", "cutoff"]], on="customer_id", how="inner")


def build_pa_naive(apps):
    pa = _with_cutoff(load("previous_applications"), apps)
    pa["days_before_cutoff"] = days_between(pa["cutoff"], pa["decision_datetime"])
    pa["approved_ratio_amt"] = pa["approved_amount"] / pa["requested_amount"]
    num = ["requested_amount", "term_months", "approved_amount", "annual_interest_rate",
           "scheduled_monthly_installment", "days_before_cutoff", "approved_ratio_amt"]
    agg = _num_agg(pa, num)
    agg["count"] = pa.groupby("customer_id").size()
    dec = pd.crosstab(pa["customer_id"], pa["decision"])
    for d in ["approved", "rejected", "cancelled"]:
        agg[f"n_{d}"] = dec[d]
        agg[f"share_{d}"] = dec[d] / agg["count"]
    return by_customer(agg, apps, "pa_")


def build_rp_naive(apps):
    rp = _with_cutoff(load("repayment_history"), apps)
    rp["unpaid"] = rp["amount_due"] - rp["amount_paid"]
    rp["paid_ratio"] = rp["amount_paid"] / rp["amount_due"]
    rp["due_days_before_cutoff"] = days_between(rp["cutoff"], rp["due_date"])
    num = ["amount_due", "amount_paid", "unpaid", "paid_ratio", "days_past_due",
           "max_days_past_due", "due_days_before_cutoff"]
    agg = _num_agg(rp, num)
    g = rp.groupby("customer_id")
    agg["n_installments"] = g.size()
    agg["n_loans"] = g["loan_id"].nunique()
    agg["sum_due"] = g["amount_due"].sum()
    agg["sum_paid"] = g["amount_paid"].sum()
    agg["sum_unpaid"] = g["unpaid"].sum()
    agg["paid_ratio_total"] = agg["sum_paid"] / agg["sum_due"]
    return by_customer(agg, apps, "rp_")


_BU_NUM = ["score", "active_accounts", "outstanding_balance", "revolving_credit_limit",
           "revolving_outstanding_balance", "utilization", "inquiries_last_6m",
           "delinquent_accounts_last_12m"]


def build_bu_naive(apps):
    """Laporan biro untuk aplikasi saat ini: pivot per provider + agregat lintas provider."""
    bo = load("bureau_observations")
    cur = bo[bo["application_id"].isin(apps["application_id"])].merge(
        cutoffs(apps)[["application_id", "cutoff"]], on="application_id")
    cur["report_age_days"] = days_between(cur["cutoff"], cur["report_datetime"])
    num = _BU_NUM + ["report_age_days"]
    piv = cur.pivot(index="application_id", columns="bureau_provider", values=num)
    piv.columns = [f"{c}_{p[-1]}" for c, p in piv.columns]
    agg = cur.groupby("application_id")[num].agg(STATS)
    agg.columns = [f"{c}_{s}" for c, s in agg.columns]
    agg["n_providers"] = cur.groupby("application_id").size()
    agg["n_unscored"] = cur["score"].isna().groupby(cur["application_id"]).sum()
    out = piv.join(agg).reindex(apps["application_id"])
    return finalize(out, "bu_")


def build_bp_naive(apps):
    """Laporan biro untuk previous applications, diagregasi per customer."""
    bo = load("bureau_observations")
    prev = _with_cutoff(bo[~bo["application_id"].isin(apps["application_id"])], apps)
    prev["report_age_days"] = days_between(prev["cutoff"], prev["report_datetime"])
    agg = _num_agg(prev, _BU_NUM + ["report_age_days"], ["mean", "max", "min"])
    agg["n_reports"] = prev.groupby("customer_id").size()
    return by_customer(agg, apps, "bp_")


def build_ab_naive(apps):
    ab = load("account_behavior")
    ab["net_flow"] = ab["inflows"] - ab["outflows"]
    num = ["opening_balance", "closing_balance", "inflows", "outflows", "income_like_credits",
           "average_balance", "minimum_balance", "low_balance_days", "net_flow"]
    agg = _num_agg(ab, num)
    g = ab.groupby("customer_id")
    agg["n_months"] = g.size()
    last = ab.sort_values("month").groupby("customer_id").last()
    for c in ["closing_balance", "average_balance", "income_like_credits", "low_balance_days"]:
        agg[f"{c}_last"] = last[c]
    cut = cutoffs(apps).set_index("customer_id")["cutoff"]
    agg["days_since_last_month"] = days_between(cut.reindex(agg.index), last["month"])
    return by_customer(agg, apps, "ab_")


def build_ec_naive(apps):
    cols = ["order_id", "customer_id", "transaction_datetime", "product_category", "quantity",
            "item_total", "order_total", "payment_method", "order_status"]
    ec = load("ecommerce_transactions", usecols=cols)
    ec["quantity"] = ec["quantity"].astype("int16")
    ec["item_total"] = ec["item_total"].astype("int64")
    completed = ec["order_status"] == "completed"

    # Level order: order_total berulang per item -> dedup per order_id.
    orders = ec.drop_duplicates("order_id")[["order_id", "customer_id", "transaction_datetime",
                                             "order_total", "payment_method", "order_status"]]
    orders = _with_cutoff(orders, apps)
    orders["completed"] = orders["order_status"] == "completed"
    done = orders[orders["completed"]]
    g = orders.groupby("customer_id")
    agg = pd.DataFrame({"n_orders": g.size(), "n_completed": g["completed"].sum()})
    agg["cancel_ratio"] = 1 - agg["n_completed"] / agg["n_orders"]
    gd = done.groupby("customer_id")["order_total"]
    agg["spend_sum"] = gd.sum()
    agg["spend_mean"] = gd.mean()
    agg["spend_max"] = gd.max()
    agg["spend_std"] = gd.std()
    agg["days_since_last_order"] = days_between(g["cutoff"].first(), g["transaction_datetime"].max())
    agg["days_since_first_order"] = days_between(g["cutoff"].first(), g["transaction_datetime"].min())
    agg["spend_per_30d"] = agg["spend_sum"] / (agg["days_since_first_order"].clip(lower=30) / 30)
    pm = pd.crosstab(orders["customer_id"], orders["payment_method"], normalize="index")
    agg = agg.join(pm.add_prefix("pm_share_"))

    # Level item: proporsi nilai belanja completed per kategori.
    items = ec[completed]
    cat = items.pivot_table(index="customer_id", columns="product_category", values="item_total",
                            aggfunc="sum", fill_value=0, observed=False)
    cat = cat.div(cat.sum(axis=1), axis=0).add_prefix("cat_share_")
    agg = agg.join(cat)
    agg["n_items"] = ec.groupby("customer_id").size()
    agg["qty_mean"] = items.groupby("customer_id")["quantity"].mean()
    return by_customer(agg, apps, "ec_")

### 5.3 Riwayat cicilan (`rp_`)

Fitur riwayat cicilan (rp_). Sinyal terkuat di EDA.

Semantik kolom (diverifikasi di data):
- days_past_due     = DPD cicilan saat cutoff (0 jika sudah lunas)
- max_days_past_due = DPD terburuk cicilan itu (= keterlambatan bayar untuk cicilan lunas)

In [ ]:
RP_WINDOWS = [90, 180, 365]


def _load_rp(apps):
    rp = load("repayment_history").merge(cutoffs(apps)[["customer_id", "cutoff"]],
                                         on="customer_id", how="inner")
    rp["age_days"] = days_between(rp["cutoff"], rp["due_date"])
    rp["unpaid"] = rp["amount_due"] - rp["amount_paid"]
    rp["paid_ratio"] = rp["amount_paid"] / rp["amount_due"]
    rp["pay_delay_days"] = days_between(rp["last_payment_datetime"], rp["due_date"])
    rp.loc[rp["unpaid"] > 0, "pay_delay_days"] = np.nan  # belum lunas: keterlambatan belum final
    for t in [1, 30, 60, 90]:
        rp[f"dpd{t}"] = (rp["max_days_past_due"] >= t).astype("int8")
    rp["late_now"] = (rp["days_past_due"] > 0).astype("int8")
    return rp


def build_rp_core(apps):
    """E10: DPD dan pembayaran all-time + kondisi saat cutoff."""
    rp = _load_rp(apps)
    g = rp.groupby("customer_id")
    agg = pd.DataFrame(index=g.size().index)
    for t in [1, 30, 60, 90]:
        agg[f"n_dpd{t}"] = g[f"dpd{t}"].sum()
        agg[f"share_dpd{t}"] = g[f"dpd{t}"].mean()
    agg["maxdpd_median"] = g["max_days_past_due"].median()
    agg["maxdpd_q75"] = g["max_days_past_due"].quantile(0.75)
    agg["maxdpd_sum"] = g["max_days_past_due"].sum()
    agg["n_late_now"] = g["late_now"].sum()
    agg["dpd_now_max"] = g["days_past_due"].max()
    agg["dpd_now_sum"] = g["days_past_due"].sum()
    agg["arrears_now"] = g["unpaid"].sum()
    agg["arrears_ratio"] = agg["arrears_now"] / g["amount_due"].sum()
    agg["delay_mean"] = g["pay_delay_days"].mean()
    agg["delay_max"] = g["pay_delay_days"].max()
    agg["delay_std"] = g["pay_delay_days"].std()
    agg["n_unpaid_inst"] = (rp["unpaid"] > 0).groupby(rp["customer_id"]).sum()
    agg["hist_len_days"] = g["age_days"].max()
    return by_customer(agg, apps, "rp_")


def build_rp_window(apps):
    """E11: jendela 90/180/365 hari (berdasarkan due_date) + recency kejadian buruk."""
    rp = _load_rp(apps)
    agg = pd.DataFrame(index=rp["customer_id"].unique())
    for w in RP_WINDOWS:
        sub = rp[rp["age_days"] <= w]
        g = sub.groupby("customer_id")
        agg[f"w{w}_n"] = g.size()
        agg[f"w{w}_maxdpd_max"] = g["max_days_past_due"].max()
        agg[f"w{w}_maxdpd_mean"] = g["max_days_past_due"].mean()
        agg[f"w{w}_share_dpd1"] = g["dpd1"].mean()
        agg[f"w{w}_share_dpd30"] = g["dpd30"].mean()
        agg[f"w{w}_paid_ratio"] = g["amount_paid"].sum() / g["amount_due"].sum()
        agg[f"w{w}_delay_mean"] = g["pay_delay_days"].mean()
    for t in [1, 30, 90]:
        last = rp[rp[f"dpd{t}"] == 1].groupby("customer_id")["age_days"].min()
        agg[f"days_since_dpd{t}"] = last
    agg["days_since_last_payment"] = days_between(
        rp.groupby("customer_id")["cutoff"].first(),
        rp.groupby("customer_id")["last_payment_datetime"].max())
    agg["days_since_last_due"] = rp.groupby("customer_id")["age_days"].min()
    return by_customer(agg, apps, "rp_")


def build_rp_loan(apps):
    """E12: dua tingkat cicilan -> pinjaman -> customer. ever90 per pinjaman = analog target."""
    rp = _load_rp(apps)
    lg = rp.groupby(["customer_id", "loan_id"])
    loan = pd.DataFrame({
        "maxdpd": lg["max_days_past_due"].max(),
        "paid_ratio": lg["amount_paid"].sum() / lg["amount_due"].sum(),
        "n_inst": lg.size(),
        "first_due_age": lg["age_days"].max(),
        "last_due_age": lg["age_days"].min(),
        "share_dpd30": lg["dpd30"].mean(),
        "principal_proxy": lg["amount_due"].first(),
    }).reset_index()
    for t in [30, 60, 90]:
        loan[f"ever{t}"] = (loan["maxdpd"] >= t).astype("int8")
    g = loan.groupby("customer_id")
    agg = pd.DataFrame({"n_loans_obs": g.size()})
    for t in [30, 60, 90]:
        agg[f"n_loans_ever{t}"] = g[f"ever{t}"].sum()
        agg[f"share_loans_ever{t}"] = g[f"ever{t}"].mean()
    agg["loan_maxdpd_mean"] = g["maxdpd"].mean()
    agg["loan_maxdpd_min"] = g["maxdpd"].min()
    agg["loan_paid_ratio_min"] = g["paid_ratio"].min()
    agg["loan_share_dpd30_max"] = g["share_dpd30"].max()
    # Pinjaman paling baru (jatuh tempo pertama paling dekat ke cutoff).
    recent = loan.sort_values("first_due_age").groupby("customer_id").first()
    for c in ["maxdpd", "paid_ratio", "n_inst", "share_dpd30", "ever90"]:
        agg[f"recent_loan_{c}"] = recent[c]
    # Pinjaman ever90 paling baru: seberapa lama sejak kejadian default terakhir.
    agg["days_since_ever90_loan"] = loan[loan["ever90"] == 1].groupby("customer_id")[
        "last_due_age"].min()
    return by_customer(agg, apps, "rp_")


def _longest_streak(flags):
    best = cur = 0
    for f in flags:
        cur = cur + 1 if f else 0
        best = max(best, cur)
    return best


def build_rp_trend(apps):
    """E13: arah perubahan perilaku bayar, DPD berbobot waktu, streak terlambat terpanjang."""
    rp = _load_rp(apps)
    rp["w"] = np.exp(-rp["age_days"] / 180)
    rp["w_maxdpd"] = rp["w"] * rp["max_days_past_due"]
    rp["w_dpd30"] = rp["w"] * rp["dpd30"]
    g = rp.groupby("customer_id")
    agg = pd.DataFrame({
        "wdpd_sum": g["w_maxdpd"].sum(),
        "wdpd_mean": g["w_maxdpd"].sum() / g["w"].sum(),
        "wdpd30_share": g["w_dpd30"].sum() / g["w"].sum(),
    })
    recent = rp[rp["age_days"] <= 180].groupby("customer_id")["max_days_past_due"]
    prior = rp[(rp["age_days"] > 180) & (rp["age_days"] <= 360)].groupby("customer_id")[
        "max_days_past_due"]
    agg["maxdpd_6m_minus_prev6m"] = recent.max() - prior.max()
    agg["meandpd_6m_minus_prev6m"] = recent.mean() - prior.mean()
    # Kemiringan max DPD terhadap umur cicilan (negatif umur = makin baru).
    x = -rp["age_days"]
    rp["_x"], rp["_xy"], rp["_xx"] = x, x * rp["max_days_past_due"], x * x
    s = rp.groupby("customer_id")[["_x", "_xy", "_xx", "max_days_past_due"]].sum()
    n = g.size()
    den = n * s["_xx"] - s["_x"] ** 2
    agg["maxdpd_slope"] = (n * s["_xy"] - s["_x"] * s["max_days_past_due"]) / den.replace(0, np.nan)
    rp = rp.sort_values(["customer_id", "due_date"])
    agg["longest_late_streak"] = rp.groupby("customer_id")["dpd1"].agg(_longest_streak)
    return by_customer(agg, apps, "rp_")

### 5.4 Biro kredit (`bu_`, `bp_`)

Fitur biro kredit: bu_ (laporan aplikasi saat ini), bp_ (laporan previous applications).

Distribusi skor ketiga provider identik, jadi tidak ada normalisasi per provider.

In [ ]:
def _split(apps):
    bo = load("bureau_observations")
    cur_mask = bo["application_id"].isin(apps["application_id"])
    cur = bo[cur_mask].merge(cutoffs(apps)[["application_id", "cutoff"]], on="application_id")
    prev = bo[~cur_mask].merge(cutoffs(apps)[["customer_id", "cutoff"]], on="customer_id")
    for d in (cur, prev):
        d["report_age_days"] = days_between(d["cutoff"], d["report_datetime"])
        d["avail_age_days"] = days_between(d["cutoff"], d["available_datetime"])
    return cur, prev


def build_bu_extra(apps):
    """E20: utilisasi agregat, sebaran skor antar provider, laporan paling segar."""
    cur, _ = _split(apps)
    g = cur.groupby("application_id")
    agg = pd.DataFrame({
        "util_total": g["revolving_outstanding_balance"].sum()
        / g["revolving_credit_limit"].sum().replace(0, np.nan),
        "has_revolving": (g["revolving_credit_limit"].max() > 0).astype("int8"),
        "score_range": g["score"].max() - g["score"].min(),
        "outstanding_sum": g["outstanding_balance"].sum(),
        "revolving_headroom": g["revolving_credit_limit"].max()
        - g["revolving_outstanding_balance"].max(),
        "accounts_sum": g["active_accounts"].sum(),
        "inquiries_sum": g["inquiries_last_6m"].sum(),
        "delinq_sum": g["delinquent_accounts_last_12m"].sum(),
        "n_delinq_providers": (cur["delinquent_accounts_last_12m"] > 0).groupby(
            cur["application_id"]).sum(),
        "avail_age_min": g["avail_age_days"].min(),
    })
    fresh = cur.sort_values("report_age_days").groupby("application_id").first()
    for c in ["score", "utilization", "inquiries_last_6m", "delinquent_accounts_last_12m",
              "outstanding_balance", "report_age_days"]:
        agg[f"fresh_{c}"] = fresh[c]
    # Skor dengan bobot kesegaran laporan.
    w = np.exp(-cur["report_age_days"] / 60)
    sw = (cur["score"] * w).groupby(cur["application_id"]).sum()
    ww = w.where(cur["score"].notna(), 0).groupby(cur["application_id"]).sum()
    agg["score_fresh_weighted"] = sw / ww.replace(0, np.nan)
    return finalize(agg.reindex(apps["application_id"]), "bu_")


def build_bp_trend(apps):
    """E21: perubahan skor/utang/inquiry dari laporan previous applications ke laporan saat ini."""
    cur, prev = _split(apps)
    cur = cur.merge(apps[["application_id", "customer_id"]], on=["application_id", "customer_id"])
    num = ["score", "outstanding_balance", "inquiries_last_6m", "delinquent_accounts_last_12m",
           "active_accounts", "utilization"]
    # Laporan sebelumnya terakhir per (customer, provider) -> delta terhadap provider yang sama.
    last_prev = prev.sort_values("report_datetime").groupby(
        ["customer_id", "bureau_provider"], observed=True)[num + ["report_age_days"]].last()
    c = cur.set_index(["customer_id", "bureau_provider"])[num]
    delta = (c - last_prev[num].reindex(c.index)).groupby(level=0).mean()
    agg = delta.add_suffix("_delta_mean")
    agg["score_delta_min"] = (c["score"] - last_prev["score"].reindex(c.index)).groupby(
        level=0).min()
    first_prev = prev.sort_values("report_datetime").groupby("customer_id")["score"].first()
    agg["score_change_since_first"] = cur.groupby("customer_id")["score"].mean() - first_prev
    agg["bureau_hist_days"] = prev.groupby("customer_id")["report_age_days"].max()
    agg["n_prev_reports"] = prev.groupby("customer_id").size()
    agg["prev_delinq_max"] = prev.groupby("customer_id")["delinquent_accounts_last_12m"].max()
    agg["prev_score_min"] = prev.groupby("customer_id")["score"].min()
    # Kemiringan skor per tahun terhadap waktu laporan (prev + current).
    allr = pd.concat([prev[["customer_id", "report_age_days", "score"]],
                      cur[["customer_id", "report_age_days", "score"]]]).dropna()
    x = -allr["report_age_days"] / 365
    allr = allr.assign(_x=x, _xy=x * allr["score"], _xx=x * x)
    s = allr.groupby("customer_id")[["_x", "_xy", "_xx", "score"]].sum()
    n = allr.groupby("customer_id").size()
    den = n * s["_xx"] - s["_x"] ** 2
    agg["score_slope_per_year"] = (n * s["_xy"] - s["_x"] * s["score"]) / den.where(den > 1e-9)
    return by_customer(agg, apps, "bp_")

### 5.5 Aplikasi sebelumnya (`pa_`)

Fitur aplikasi sebelumnya (pa_).

In [ ]:
PA_WINDOWS = [30, 90, 180, 365]


def _load_pa(apps):
    pa = load("previous_applications").merge(
        apps[["customer_id", "decision_datetime", "loan_amount", "annual_interest_rate",
              "proposed_monthly_installment"]].rename(columns={
                  "decision_datetime": "cutoff", "loan_amount": "cur_amount",
                  "annual_interest_rate": "cur_rate",
                  "proposed_monthly_installment": "cur_inst"}),
        on="customer_id", how="inner")
    pa["age_days"] = days_between(pa["cutoff"], pa["decision_datetime"])
    pa["app_age_days"] = days_between(pa["cutoff"], pa["application_datetime"])
    return pa


def build_pa_window(apps):
    """E30: hitungan per jendela, recency per keputusan, kecepatan, pinjaman baru belum jatuh tempo."""
    pa = _load_pa(apps)
    agg = pd.DataFrame(index=pa["customer_id"].unique())
    for w in PA_WINDOWS:
        sub = pa[pa["app_age_days"] <= w]
        ct = pd.crosstab(sub["customer_id"], sub["decision"])
        agg[f"w{w}_n_apps"] = sub.groupby("customer_id").size()
        for d in ["approved", "rejected", "cancelled"]:
            agg[f"w{w}_n_{d}"] = ct[d] if d in ct else 0
    wcols = [c for c in agg.columns if c.startswith("w")]
    agg[wcols] = agg[wcols].fillna(0)
    for d in ["approved", "rejected", "cancelled"]:
        agg[f"days_since_last_{d}"] = pa[pa["decision"] == d].groupby("customer_id")[
            "age_days"].min()
    agg["days_since_last_app"] = pa.groupby("customer_id")["app_age_days"].min()
    agg["hist_len_days"] = pa.groupby("customer_id")["app_age_days"].max()
    agg["apps_per_year"] = pa.groupby("customer_id").size() / (agg["hist_len_days"] / 365).clip(
        lower=1 / 12)
    pa_s = pa.sort_values(["customer_id", "application_datetime"])
    pa_s["gap"] = pa_s.groupby("customer_id")["app_age_days"].diff().abs()
    agg["app_gap_mean"] = pa_s.groupby("customer_id")["gap"].mean()
    agg["app_gap_min"] = pa_s.groupby("customer_id")["gap"].min()
    agg["process_days_mean"] = (pa["app_age_days"] - pa["age_days"]).groupby(
        pa["customer_id"]).mean()
    # Pinjaman cair <= 31 hari sebelum cutoff belum punya cicilan jatuh tempo (target 19,1%).
    rp_loans = load("repayment_history", usecols=["loan_id"])["loan_id"].unique()
    appr = pa[pa["decision"] == "approved"]
    fresh = appr[~appr["loan_id"].isin(rp_loans)]
    agg["n_fresh_loans"] = fresh.groupby("customer_id").size()
    agg["n_fresh_loans"] = agg["n_fresh_loans"].fillna(0)
    agg["fresh_loan_amount"] = fresh.groupby("customer_id")["approved_amount"].sum()
    agg["fresh_loan_inst"] = fresh.groupby("customer_id")["scheduled_monthly_installment"].sum()
    last = pa.sort_values("application_datetime").groupby("customer_id").last()
    agg["last_decision_rejected"] = (last["decision"] == "rejected").astype("float32")
    agg["last_decision_cancelled"] = (last["decision"] == "cancelled").astype("float32")
    return by_customer(agg, apps, "pa_")


def build_pa_burden(apps):
    """E31: beban cicilan aktif, nominal & bunga dibanding pinjaman saat ini."""
    pa = _load_pa(apps)
    appr = pa[pa["decision"] == "approved"].copy()
    appr["end"] = appr["decision_datetime"] + pd.to_timedelta(appr["term_months"] * 30.44, unit="D")
    appr["active"] = appr["end"] > appr["cutoff"]
    act = appr[appr["active"]]
    g = act.groupby("customer_id")
    agg = pd.DataFrame(index=pa["customer_id"].unique())
    agg["n_active_loans"] = g.size()
    agg["active_inst_sum"] = g["scheduled_monthly_installment"].sum()
    agg["active_amount_sum"] = g["approved_amount"].sum()
    agg["active_months_left_sum"] = (days_between(act["end"], act["cutoff"]) / 30.44).groupby(
        act["customer_id"]).sum()
    agg[["n_active_loans", "active_inst_sum", "active_amount_sum"]] = agg[
        ["n_active_loans", "active_inst_sum", "active_amount_sum"]].fillna(0)
    cur = pa.groupby("customer_id")[["cur_amount", "cur_rate", "cur_inst"]].first()
    ga = appr.groupby("customer_id")
    agg["amount_vs_max_approved"] = cur["cur_amount"] / ga["approved_amount"].max()
    agg["amount_vs_mean_requested"] = cur["cur_amount"] / pa.groupby("customer_id")[
        "requested_amount"].mean()
    last_appr = appr.sort_values("decision_datetime").groupby("customer_id").last()
    agg["amount_vs_last_approved"] = cur["cur_amount"] / last_appr["approved_amount"]
    agg["rate_minus_last"] = cur["cur_rate"] - last_appr["annual_interest_rate"]
    agg["rate_minus_mean"] = cur["cur_rate"] - ga["annual_interest_rate"].mean()
    agg["rate_prev_max"] = ga["annual_interest_rate"].max()
    agg["inst_vs_last"] = cur["cur_inst"] / last_appr["scheduled_monthly_installment"]
    agg["approved_share_amt"] = ga["approved_amount"].sum() / pa.groupby("customer_id")[
        "requested_amount"].sum()
    agg["rejected_amount_max"] = pa[pa["decision"] == "rejected"].groupby("customer_id")[
        "requested_amount"].max()
    agg["total_inst_with_current"] = agg["active_inst_sum"] + cur["cur_inst"]
    return by_customer(agg, apps, "pa_")

### 5.6 Perilaku rekening (`ab_`)

Fitur perilaku rekening (ab_). Agregat mentahnya lemah, jadi fokus ke tren, volatilitas, rasio.

In [ ]:
_NUM = ["closing_balance", "inflows", "outflows", "income_like_credits", "average_balance",
        "minimum_balance", "low_balance_days", "net_flow"]


def _load_ab():
    ab = load("account_behavior")
    ab["net_flow"] = ab["inflows"] - ab["outflows"]
    ab = ab.sort_values(["customer_id", "month"])
    # Urutan bulan dari belakang: 1 = bulan observasi terakhir.
    ab["rev_idx"] = ab.groupby("customer_id").cumcount(ascending=False) + 1
    return ab


def _slope(df, col):
    """Kemiringan OLS col terhadap urutan bulan, dinormalisasi oleh rata-rata |col|."""
    x = -df["rev_idx"].astype(float)
    t = df.assign(_x=x, _y=df[col], _xy=x * df[col], _xx=x * x)
    s = t.groupby("customer_id")[["_x", "_y", "_xy", "_xx"]].sum()
    n = t.groupby("customer_id").size()
    den = n * s["_xx"] - s["_x"] ** 2
    slope = (n * s["_xy"] - s["_x"] * s["_y"]) / den.where(den > 0)
    return slope / (s["_y"].abs() / n).replace(0, np.nan)


def build_ab_window(apps):
    """E40: jendela 3/6/12 bulan terakhir, tren, volatilitas, rasio terbaru vs sebelumnya."""
    ab = _load_ab()
    agg = pd.DataFrame(index=ab["customer_id"].unique())
    for w in [3, 6, 12]:
        g = ab[ab["rev_idx"] <= w].groupby("customer_id")
        for c in ["average_balance", "minimum_balance", "income_like_credits", "net_flow"]:
            agg[f"w{w}_{c}_mean"] = g[c].mean()
        agg[f"w{w}_min_balance_min"] = g["minimum_balance"].min()
        agg[f"w{w}_low_days_sum"] = g["low_balance_days"].sum()
        agg[f"w{w}_n_neg_flow"] = (ab.loc[ab["rev_idx"] <= w, "net_flow"] < 0).groupby(
            ab.loc[ab["rev_idx"] <= w, "customer_id"]).sum()
        agg[f"w{w}_outflow_inflow"] = g["outflows"].sum() / g["inflows"].sum().replace(0, np.nan)
    prev = ab[(ab["rev_idx"] > 3) & (ab["rev_idx"] <= 6)].groupby("customer_id")
    for c in ["average_balance", "income_like_credits", "low_balance_days"]:
        agg[f"{c}_3m_vs_prev3m"] = agg.get(f"w3_{c}_mean", ab[ab["rev_idx"] <= 3].groupby(
            "customer_id")[c].mean()) - prev[c].mean()
    for c in ["average_balance", "income_like_credits", "closing_balance", "outflows"]:
        agg[f"{c}_slope"] = _slope(ab, c)
    g = ab.groupby("customer_id")
    agg["income_cv"] = g["income_like_credits"].std() / g["income_like_credits"].mean().replace(
        0, np.nan)
    agg["balance_cv"] = g["average_balance"].std() / g["average_balance"].mean().abs().replace(
        0, np.nan)
    agg["share_months_low"] = (ab["low_balance_days"] > 0).groupby(ab["customer_id"]).mean()
    agg["last_low_days"] = g["low_balance_days"].last()
    agg["last_min_balance"] = g["minimum_balance"].last()
    return by_customer(agg, apps, "ab_")


def build_ab_income(apps):
    """E41: verifikasi pendapatan dan bantalan likuiditas terhadap cicilan baru."""
    ab = _load_ab()
    prof = load("customer_profiles", usecols=["customer_id", "declared_monthly_income"])
    cur = apps[["customer_id", "proposed_monthly_installment"]].merge(prof, on="customer_id")
    cur = cur.set_index("customer_id")
    agg = pd.DataFrame(index=ab["customer_id"].unique())
    for w, lab in [(3, "w3"), (100, "all")]:
        g = ab[ab["rev_idx"] <= w].groupby("customer_id")
        inc = g["income_like_credits"].mean()
        agg[f"{lab}_income_vs_declared"] = inc / cur["declared_monthly_income"].reindex(inc.index)
        agg[f"{lab}_inst_vs_income"] = cur["proposed_monthly_installment"].reindex(
            inc.index) / inc.replace(0, np.nan)
        agg[f"{lab}_avgbal_vs_inst"] = g["average_balance"].mean() / cur[
            "proposed_monthly_installment"].reindex(inc.index)
        agg[f"{lab}_minbal_vs_inst"] = g["minimum_balance"].mean() / cur[
            "proposed_monthly_installment"].reindex(inc.index)
        agg[f"{lab}_netflow_vs_inst"] = g["net_flow"].mean() / cur[
            "proposed_monthly_installment"].reindex(inc.index)
        agg[f"{lab}_outflow_vs_income"] = g["outflows"].mean() / inc.replace(0, np.nan)
    return by_customer(agg, apps, "ab_")

### 5.7 E-commerce (`ec_`)

Fitur e-commerce (ec_). Recency belanja adalah fitur tunggal terkuat di E03 (AUC 0,689).

order_total berulang per item -> metrik level order memakai data yang di-dedup per order_id.
Order cancelled bukan spending.

In [ ]:
EC_WINDOWS = [3, 7, 14, 30, 90, 180, 365]
_COLS = ["order_id", "customer_id", "transaction_datetime", "product_category", "item_total",
         "order_total", "payment_method", "order_status"]


def _load_ec(apps):
    ec = load("ecommerce_transactions", usecols=_COLS)
    ec = ec.merge(cutoffs(apps)[["customer_id", "cutoff"]], on="customer_id", how="inner")
    ec["age_days"] = days_between(ec["cutoff"], ec["transaction_datetime"]).astype("float32")
    ec["completed"] = (ec["order_status"] == "completed").astype("int8")
    orders = ec.drop_duplicates("order_id")[["customer_id", "transaction_datetime", "age_days",
                                             "order_total", "payment_method", "completed"]]
    return ec, orders


def build_ec_window(apps):
    """E50: RFM per jendela 3–365 hari, recency per status, jarak antarorder."""
    _, orders = _load_ec(apps)
    done = orders[orders["completed"] == 1]
    canc = orders[orders["completed"] == 0]
    agg = pd.DataFrame(index=orders["customer_id"].unique())
    tot_days = orders.groupby("customer_id")["age_days"].max().clip(lower=30)
    spend_all = done.groupby("customer_id")["order_total"].sum()
    n_all = orders.groupby("customer_id").size()
    for w in EC_WINDOWS:
        o = orders[orders["age_days"] <= w]
        d = done[done["age_days"] <= w]
        agg[f"w{w}_n_orders"] = o.groupby("customer_id").size()
        agg[f"w{w}_spend"] = d.groupby("customer_id")["order_total"].sum()
        agg[f"w{w}_n_cancel"] = (o["completed"] == 0).groupby(o["customer_id"]).sum()
        # Intensitas relatif terhadap rata-rata historis customer (1 = normal).
        agg[f"w{w}_order_intensity"] = agg[f"w{w}_n_orders"] / (n_all * w / tot_days)
        agg[f"w{w}_spend_intensity"] = agg[f"w{w}_spend"] / (spend_all * w / tot_days)
    win_cols = [c for c in agg.columns if c.endswith(("_n_orders", "_spend", "_n_cancel"))]
    agg[win_cols] = agg[win_cols].fillna(0)
    agg["days_since_last_completed"] = done.groupby("customer_id")["age_days"].min()
    agg["days_since_last_cancel"] = canc.groupby("customer_id")["age_days"].min()
    # Jarak antarorder: gap terakhir dibanding gap rata-rata.
    o = orders.sort_values(["customer_id", "age_days"])
    o["gap"] = o.groupby("customer_id")["age_days"].diff()
    g = o.groupby("customer_id")["gap"]
    agg["gap_mean"] = g.mean()
    agg["gap_median"] = g.median()
    agg["gap_min"] = g.min()
    recency = orders.groupby("customer_id")["age_days"].min()
    agg["recency_over_gap"] = recency / agg["gap_mean"]
    agg["n_orders_last_day"] = orders[orders["age_days"] <= 1.5].groupby("customer_id").size()
    agg["n_orders_last_day"] = agg["n_orders_last_day"].fillna(0)
    agg["last_order_total"] = orders.sort_values("age_days").groupby("customer_id")[
        "order_total"].first()
    agg["last_order_completed"] = orders.sort_values("age_days").groupby("customer_id")[
        "completed"].first()
    return by_customer(agg, apps, "ec_")


def build_ec_mix(apps):
    """E51: pergeseran metode bayar & kategori menjelang aplikasi, pola waktu belanja."""
    ec, orders = _load_ec(apps)
    agg = pd.DataFrame(index=orders["customer_id"].unique())
    pm_all = pd.crosstab(orders["customer_id"], orders["payment_method"], normalize="index")
    for w in [30, 90]:
        o = orders[orders["age_days"] <= w]
        pm = pd.crosstab(o["customer_id"], o["payment_method"], normalize="index")
        pm = pm.reindex(columns=pm_all.columns)
        agg = agg.join((pm - pm_all.reindex(pm.index)).add_prefix(f"w{w}_pm_shift_"))
    done = orders[orders["completed"] == 1]
    agg = agg.join(pd.crosstab(done["customer_id"], done["payment_method"],
                               values=done["order_total"], aggfunc="sum",
                               normalize="index").add_prefix("pm_value_share_"))
    items = ec[ec["completed"] == 1]
    cat_all = items.pivot_table(index="customer_id", columns="product_category",
                                values="item_total", aggfunc="sum", fill_value=0, observed=False)
    cat_all = cat_all.div(cat_all.sum(axis=1), axis=0)
    it30 = items[items["age_days"] <= 30]
    cat30 = it30.pivot_table(index="customer_id", columns="product_category",
                             values="item_total", aggfunc="sum", fill_value=0, observed=False)
    cat30 = cat30.div(cat30.sum(axis=1), axis=0)
    agg = agg.join((cat30 - cat_all.reindex(cat30.index)).add_prefix("w30_cat_shift_"))
    # Belanja elektronik & kesehatan 30 hari terakhir (nilai absolut).
    for c in ["electronics", "health"]:
        agg[f"w30_spend_{c}"] = it30[it30["product_category"] == c].groupby("customer_id")[
            "item_total"].sum()
        agg[f"w30_spend_{c}"] = agg[f"w30_spend_{c}"].fillna(0)
    hour = orders["transaction_datetime"].dt.hour
    orders = orders.assign(night=((hour >= 0) & (hour < 5)).astype("int8"),
                           weekend=(orders["transaction_datetime"].dt.dayofweek >= 5).astype("int8"))
    g = orders.groupby("customer_id")
    agg["share_night"] = g["night"].mean()
    agg["share_weekend"] = g["weekend"].mean()
    agg["cancel_ratio_w90"] = 1 - orders[orders["age_days"] <= 90].groupby("customer_id")[
        "completed"].mean()
    return by_customer(agg, apps, "ec_")

### 5.8 Fitur lintas tabel (`x_`)

Fitur lintas tabel (x_). Dibangun dari cache grup lain agar tidak menghitung ulang.

In [ ]:
def _get(apps, name):
    # Import lokal untuk menghindari import melingkar dengan registry.
    return cached(name, GROUPS[name], apps)


def build_x_cross(apps):
    """E60: total beban cicilan, utang biro vs pendapatan, tiga sumber pendapatan, skor stres."""
    prof = _get(apps, "prof_raw_v1")
    app_ = _get(apps, "app_raw_v1")
    bu = _get(apps, "bu_naive_v1")
    pab = _get(apps, "pa_burden_v1")
    abi = _get(apps, "ab_naive_v1")
    ec = _get(apps, "ec_naive_v1")
    rpw = _get(apps, "rp_window_v1")
    inc = prof["prof_declared_monthly_income"].replace(0, np.nan)
    inst = app_["app_proposed_monthly_installment"]
    out = pd.DataFrame(index=prof.index)
    total_inst = inst + pab["pa_active_inst_sum"].fillna(0)
    out["total_pti"] = total_inst / inc
    out["bureau_debt_to_annual_income_max"] = bu["bu_outstanding_balance_max"] / (inc * 12)
    out["bureau_debt_plus_loan_to_income"] = (bu["bu_outstanding_balance_max"]
                                              + app_["app_loan_amount"]) / (inc * 12)
    acct_inc = abi["ab_income_like_credits_mean"]
    out["acct_income_vs_declared"] = acct_inc / inc
    out["ec_monthly_spend_vs_income"] = ec["ec_spend_per_30d"] / inc
    out["ec_monthly_spend_vs_acct_income"] = ec["ec_spend_per_30d"] / acct_inc.replace(0, np.nan)
    out["total_inst_vs_acct_income"] = total_inst / acct_inc.replace(0, np.nan)
    out["rate_x_pti"] = app_["app_annual_interest_rate"] * total_inst / inc
    out["rate_x_score"] = app_["app_annual_interest_rate"] * (850 - bu["bu_score_mean"])
    stress = pd.DataFrame({
        "recent_dpd30": rpw["rp_w180_share_dpd30"].fillna(0) > 0,
        "low_balance": abi["ab_low_balance_days_mean"].fillna(0) > 5,
        "high_cancel": ec["ec_cancel_ratio"].fillna(0) > 0.15,
        "bureau_delinq": bu["bu_delinquent_accounts_last_12m_max"].fillna(0) > 0,
        "shopped_last_2d": ec["ec_days_since_last_order"].fillna(99) <= 2,
    })
    out["stress_count"] = stress.sum(axis=1)
    return finalize(out, "x_")


def build_x_rate_resid(apps):
    """E61: residual bunga. Regresi TANPA target: prediksi tier bunga dari informasi risiko yang
    terlihat (biro, cicilan, profil). Residual = penilaian risiko 'privat' lender."""
    names = ["prof_raw_v1", "bu_naive_v1", "rp_naive_v1", "pa_naive_v1", "ab_naive_v1",
             "ec_naive_v1"]
    X = pd.concat([_get(apps, n) for n in names], axis=1)
    y = _get(apps, "app_raw_v1")["app_annual_interest_rate"].values
    rng = np.random.RandomState(SEED)
    fold = rng.randint(0, 5, len(y))
    pred = np.zeros(len(y))
    params = dict(objective="regression", learning_rate=0.05, num_leaves=63,
                  min_child_samples=100, feature_fraction=0.5, bagging_fraction=0.8,
                  bagging_freq=1, seed=SEED, deterministic=True, force_row_wise=True,
                  num_threads=N_THREADS, verbose=-1)
    for k in range(5):
        tr, va = fold != k, fold == k
        m = lgb.train(params, lgb.Dataset(X[tr], y[tr]), 600)
        pred[va] = m.predict(X[va])
    out = pd.DataFrame({"rate_pred": pred, "rate_resid": y - pred}, index=X.index)
    return finalize(out, "x_")

### 5.9 Rantai bayar bulanan level event (`rp_ev_`) — temuan utama eksplorasi

Fitur rantai bayar bulanan level event (rp_ev_), H01 eksplorasi 17 Okt.

Pembangkit cicilan (experiments/explore_17okt s02/s05/s06): pembayaran hanya terjadi tepat di ulang bulan due
(tanggal-jam pencairan + j bulan), dialokasikan FIFO ke cicilan tertua, tidak pernah dibayar di muka, 0-3 cicilan
per event. Event j = 1..J dengan J = jumlah ulang bulan pencairan yang < cutoff (= decision_datetime aplikasi
sekarang); setelah tenor, event masih dihitung selama ada tunggakan. X_j = jumlah cicilan yang lunas penuh di event j,
A_j = min(j, tenor) - sum X (tunggakan setelah event j). Tidak memakai target; prior Beta dari semua customer.

In [ ]:
def _events(apps):
    rp = load("repayment_history", usecols=["loan_id", "customer_id", "due_date", "amount_due", "amount_paid",
                                            "last_payment_datetime"])
    pa = load("previous_applications", usecols=["loan_id", "decision_datetime", "term_months"]).dropna(subset=["loan_id"])
    rp = rp.merge(pa, on="loan_id").merge(cutoffs(apps)[["customer_id", "cutoff"]], on="customer_id")
    rp = rp.sort_values(["loan_id", "due_date"], kind="stable").reset_index(drop=True)
    rp["k"] = rp.groupby("loan_id").cumcount() + 1
    paid = rp["amount_paid"] >= rp["amount_due"]
    lp, dis = rp["last_payment_datetime"], rp["decision_datetime"]
    rp["e_paid"] = np.where(paid, (lp.dt.year - dis.dt.year) * 12 + (lp.dt.month - dis.dt.month), np.nan)
    L = rp.groupby("loan_id", sort=True).agg(customer_id=("customer_id", "first"), dis=("decision_datetime", "first"),
                                             cutoff=("cutoff", "first"), n=("k", "max"), term=("term_months", "first"),
                                             last_e=("e_paid", "max"), n_unpaid=("e_paid", lambda s: s.isna().sum()))
    mdiff = (L.cutoff.dt.year - L.dis.dt.year) * 12 + (L.cutoff.dt.month - L.dis.dt.month)
    before = (L.dis.dt.day * 1440 + L.dis.dt.hour * 60 + L.dis.dt.minute) < (
        L.cutoff.dt.day * 1440 + L.cutoff.dt.hour * 60 + L.cutoff.dt.minute)
    j_all = (mdiff - 1 + before.astype(int)).clip(lower=0)
    J = np.where(L.n_unpaid == 0, np.maximum(L.n, L.last_e.fillna(0)), j_all).astype(int)
    L["J"] = np.minimum(J, j_all.values)
    L = L[L.J > 0]
    ev = pd.DataFrame({"loan_id": np.repeat(L.index.values, L.J.values),
                       "j": np.concatenate([np.arange(1, x + 1) for x in L.J.values])})
    x = rp[rp.e_paid.notna()].groupby(["loan_id", "e_paid"]).size().rename("X").reset_index()
    x["j"] = x.e_paid.astype(int)
    ev = ev.merge(x[["loan_id", "j", "X"]], on=["loan_id", "j"], how="left")
    ev["X"] = ev["X"].fillna(0).astype(int)
    ev = ev.merge(L[["customer_id", "dis", "cutoff", "term"]], left_on="loan_id", right_index=True)
    ev["A"] = np.minimum(ev.j, ev.term) - ev.groupby("loan_id").X.cumsum()
    ev["A_prev"] = ev.groupby("loan_id").A.shift().fillna(0).astype(int)
    ev = ev[ev.A_prev + (ev.j <= ev.term).astype(int) > 0].copy()  # event tanpa kewajiban tidak dihitung
    ev["miss"] = (ev.X == 0).astype("int8")
    cal = ev.dis.dt.year * 12 + ev.dis.dt.month - 1 + ev.j
    ev["m_before"] = (ev.cutoff.dt.year * 12 + ev.cutoff.dt.month - 1) - cal
    return ev


def _beta_prior(k, n, min_n):
    """Prior Beta method of moments (varians dikoreksi binomial) dari customer dengan n >= min_n."""
    m = n >= min_n
    r = k[m] / n[m]
    mu = r.mean()
    v = max(r.var() - np.mean(mu * (1 - mu) / n[m]), 1e-4)
    s = mu * (1 - mu) / v - 1
    return mu * s, (1 - mu) * s


def _pd_dp(q0, q1, c, steps=9):
    """P(tunggakan mencapai 3 dalam `steps` event) dari A=0; rantai: lewat bayar q, bayar dan mengejar c."""
    p = np.zeros((len(q0), 4))
    p[:, 0] = 1
    for _ in range(steps):
        nx = np.zeros_like(p)
        nx[:, 3] = p[:, 3]
        nx[:, 1] += p[:, 0] * q0
        nx[:, 0] += p[:, 0] * (1 - q0)
        for a in (1, 2):
            nx[:, a + 1] += p[:, a] * q1
            nx[:, a] += p[:, a] * (1 - q1) * (1 - c)
            nx[:, a - 1] += p[:, a] * (1 - q1) * c
        p = nx
    return p[:, 3]


def build_rp_events(apps):
    """H01: statistik rantai bayar bulanan per customer (tanpa target)."""
    ev = _events(apps)
    at0 = ev.A_prev == 0
    g = ev.groupby("customer_id")
    g0, g1 = ev[at0].groupby("customer_id"), ev[~at0].groupby("customer_id")
    S = pd.DataFrame({"n_ev": g.size(), "n_miss": g.miss.sum()})
    S = S.join(pd.DataFrame({"n0": g0.size(), "m0": g0.miss.sum()}), how="left")
    paid1 = ev[~at0 & (ev.X >= 1)]
    S = S.join(pd.DataFrame({"n1": g1.size(), "m1": g1.miss.sum()}), how="left")
    S = S.join(pd.DataFrame({"p1": paid1.groupby("customer_id").size(),
                             "c1": (paid1.X >= 2).groupby(paid1.customer_id).sum()}), how="left")
    S = S.fillna(0)
    a0, b0 = _beta_prior(S.m0.values, S.n0.values, 6)
    a1, b1 = _beta_prior(S.m1.values, S.n1.values, 4)
    am, bm = _beta_prior(S.n_miss.values, S.n_ev.values, 6)
    out = pd.DataFrame(index=S.index)
    out["n_ev"], out["n_ev_at0"], out["n_ev_behind"] = S.n_ev, S.n0, S.n1
    out["miss_post"] = (S.n_miss + am) / (S.n_ev + am + bm)
    out["q0_post"] = (S.m0 + a0) / (S.n0 + a0 + b0)
    out["q1_post"] = (S.m1 + a1) / (S.n1 + a1 + b1)
    out["catch_rate"] = (S.c1 / S.p1).where(S.p1 > 0)
    # laju mengejar dengan prior lemah (rata-rata populasi, kekuatan 4 event) supaya tetap bervariasi
    c_mu = S.c1.sum() / max(S.p1.sum(), 1)
    out["catch_post"] = (S.c1 + 4 * c_mu) / (S.p1 + 4)
    out["pd_struct"] = _pd_dp(out.q0_post.values, out.q1_post.values, out.catch_post.values)
    out["max_arrears"] = g.A.max()
    out["months_since_last_miss"] = ev[ev.miss == 1].groupby("customer_id").m_before.min()
    out["n_post_term_ev"] = (ev.j > ev.term).groupby(ev.customer_id).sum()
    for w in (3, 6, 12):
        e = ev[ev.m_before < w]
        out[f"miss_rate_{w}m"] = e.groupby("customer_id").miss.mean()
        out[f"n_ev_{w}m"] = e.groupby("customer_id").size()
    return by_customer(out, apps, "rp_ev_")

### 5.10 Bangun matriks fitur

23 grup (528 kolom, sama dengan set "all" pipeline eksperimen) + 18 kolom rantai bayar event. Kolom diurutkan
(`sorted`) persis seperti `run.build_matrix`. Grup `x_` memakai hasil grup lain lewat cache memori.

In [ ]:
GROUPS = {
    "app_raw_v1": build_app_raw, "prof_raw_v1": build_prof_raw, "app_ratios_v1": build_app_ratios,
    "pa_naive_v1": build_pa_naive, "rp_naive_v1": build_rp_naive, "bu_naive_v1": build_bu_naive,
    "bp_naive_v1": build_bp_naive, "ab_naive_v1": build_ab_naive, "ec_naive_v1": build_ec_naive,
    "rp_core_v1": build_rp_core, "rp_window_v1": build_rp_window, "rp_loan_v1": build_rp_loan,
    "rp_trend_v1": build_rp_trend, "bu_extra_v1": build_bu_extra, "bp_trend_v1": build_bp_trend,
    "pa_window_v1": build_pa_window, "pa_burden_v1": build_pa_burden, "ab_window_v1": build_ab_window,
    "ab_income_v1": build_ab_income, "ec_window_v1": build_ec_window, "ec_mix_v1": build_ec_mix,
    "x_cross_v1": build_x_cross, "x_rate_resid_v1": build_x_rate_resid,
}
EXTRA = {"rp_events_v1": build_rp_events}


def build_matrix(groups, apps):
    parts = []
    for g in groups:
        t = time.time()
        parts.append(cached(g, GROUPS[g], apps))
        print(f"  fitur {g:<16} {parts[-1].shape[1]:>4} kolom  ({(time.time() - t) / 60:.1f} mnt)", flush=True)
    X = pd.concat(parts, axis=1)
    assert X.columns.is_unique, "nama kolom duplikat antar grup"
    return X[sorted(X.columns)]


X = build_matrix(list(GROUPS), apps)
E = pd.concat([cached(n, f, apps) for n, f in EXTRA.items()], axis=1)
X = pd.concat([X, E], axis=1)
del E
gc.collect()
print("matriks fitur:", X.shape, f"({(time.time() - T0) / 60:.1f} mnt sejak awal)")

## 6. Validasi dan fitur nested model level order e-commerce

CV = StratifiedKFold 5 fold (shuffle, `random_state=42`) di train, sama untuk semua model. Fitur `ecm_*` berasal dari
model LightGBM level **order** (1 baris = 1 order, label = target customer) yang diagregasi per customer. Supaya tidak
bocor, fitur ini **nested**: untuk fold luar k, baris validasi fold k dan test diprediksi model yang dilatih di fold ≠ k;
baris latih fold j ≠ k diprediksi model yang dilatih di fold di luar {k, j}. Early stopping model order memakai 10%
customer holdout di dalam data latihnya sendiri. Hasil per fold k: `tr{k}`, `va{k}`, `te{k}`.

In [ ]:
def make_folds(train):
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
    fold = np.zeros(len(train), dtype=int)
    for k, (_, va) in enumerate(skf.split(train, train["target"])):
        fold[va] = k
    return pd.DataFrame({"application_id": train["application_id"].values, "fold": fold})


y = apps.loc[m_tr, "target"].astype(int).values
fold = make_folds(apps[m_tr])["fold"].values
test_ids = apps.loc[~m_tr, "application_id"].values
print("ukuran fold:", np.bincount(fold).tolist(), "| target rate per fold:", [round(y[fold == k].mean(), 4) for k in range(N_FOLDS)])

CATS = ["groceries", "fashion", "electronics", "home", "health", "education", "personal_care"]
FEATS = ["recency_h", "total", "total_vi", "pm", "st", "n_items", "qty", "hour", "dow"] + CATS
ROW_P = dict(objective="binary", learning_rate=0.1, num_leaves=63, min_child_samples=500,
             feature_fraction=0.8, bagging_fraction=0.7, bagging_freq=1, lambda_l2=10, verbose=-1,
             seed=SEED, deterministic=True, force_row_wise=True, num_threads=4)
AGGX = False


def build_orders(apps):
    """Tabel order dengan fitur intrinsik; cutoff = decision_datetime aplikasi customer."""
    ec = load("ecommerce_transactions")
    share = pd.crosstab(ec["order_id"], ec["product_category"], values=ec["item_total"], aggfunc="sum",
                        normalize="index").astype("float32")
    share.columns = share.columns.astype(str)
    od = ec.groupby("order_id", sort=True).agg(
        customer_id=("customer_id", "first"), t=("transaction_datetime", "first"),
        total=("order_total", "first"), pm=("payment_method", "first"), st=("order_status", "first"),
        n_items=("order_item_id", "size"), qty=("quantity", "sum"))
    del ec
    gc.collect()
    od = od.join(share[CATS]).reset_index(drop=True)
    del share
    prof = load("customer_profiles", usecols=["customer_id", "declared_monthly_income"])
    od = od.merge(apps[["customer_id", "decision_datetime"]], on="customer_id").merge(prof, on="customer_id")
    assert (od["t"] < od["decision_datetime"]).all()
    od["recency_h"] = (od["decision_datetime"] - od["t"]).dt.total_seconds() / 3600
    od["hour"] = od["t"].dt.hour
    od["dow"] = od["t"].dt.dayofweek
    od["pm"] = od["pm"].cat.codes
    od["st"] = od["st"].cat.codes
    od["total_vi"] = od["total"] / od["declared_monthly_income"]
    od = od[["customer_id"] + FEATS].sort_values(["customer_id", "recency_h"], kind="stable")
    od[FEATS] = od[FEATS].astype("float32")
    od = od.reset_index(drop=True)
    return od


def fit_rows(od, mask, y_row, seed):
    """Latih model baris pada baris mask; early stopping pada 10% customer holdout di dalamnya."""
    cust = od.loc[mask, "customer_id"].unique()
    rng = np.random.default_rng(seed)
    hold = set(rng.choice(cust, len(cust) // 10, replace=False))
    is_hold = od["customer_id"].isin(hold).values
    tr, es = mask & ~is_hold, mask & is_hold
    cats = [c for c in ("pm", "st", "cat") if c in FEATS]
    dtr = lgb.Dataset(od.loc[tr, FEATS], y_row[tr], categorical_feature=cats, free_raw_data=True)
    des = lgb.Dataset(od.loc[es, FEATS], y_row[es], reference=dtr)
    m = lgb.train({**ROW_P, "seed": seed}, dtr, 4000, valid_sets=[des],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
    return m


def aggregate(od, pred, rows, customers):
    """Agregasi prediksi order (raw) per customer -> DataFrame berindeks customer_id (urut customers)."""
    d = pd.DataFrame({"customer_id": od["customer_id"].values[rows], "p": pred,
                      "rec": od["recency_h"].values[rows]})
    d["rk"] = d.groupby("customer_id").cumcount()  # od sudah urut recency naik per customer
    g = d.groupby("customer_id")["p"]
    agg = pd.DataFrame({"mean": g.mean(), "max": g.max(), "min": g.min(), "std": g.std(),
                        "last3": d[d.rk < 3].groupby("customer_id")["p"].mean(),
                        "last10": d[d.rk < 10].groupby("customer_id")["p"].mean(),
                        "rec30": d[d.rec <= 720].groupby("customer_id")["p"].mean()})
    if AGGX:  # E121e: agregat waktu tambahan
        d["q"] = 1 / (1 + np.exp(-d["p"]))
        for hl in (168, 720):  # bobot peluruhan, half-life 7 / 30 hari
            d["w"] = 0.5 ** (d["rec"] / hl)
            d["wp"] = d["w"] * d["p"]
            s = d.groupby("customer_id")[["w", "wp"]].sum()
            agg[f"ewm{hl}"] = s["wp"] / s["w"]
        agg["rec7"] = d[d.rec <= 168].groupby("customer_id")["p"].mean()
        agg["rec90"] = d[d.rec <= 2160].groupby("customer_id")["p"].mean()
        agg["q90"] = g.quantile(0.9)
        top = d.sort_values(["customer_id", "p"], ascending=[True, False], kind="stable")
        agg["top3"] = top[top.groupby("customer_id").cumcount() < 3].groupby("customer_id")["p"].mean()
        agg["risk30"] = d[d.rec <= 720].groupby("customer_id")["q"].sum()
        agg["risk_all"] = d.groupby("customer_id")["q"].sum()
        agg["trend30"] = agg["rec30"] - agg["mean"]
    agg.columns = [f"ecm_{c}" for c in agg.columns]
    return agg.reindex(customers).astype("float32")


def build_ecm(apps, y, fold):
    """Fitur ecm_* nested per fold (sama dengan e121.py --build, hasil di memori)."""
    tr_apps = apps[apps["is_test"].eq(0).values].reset_index(drop=True)
    od = build_orders(apps)
    cfold = pd.Series(fold, index=tr_apps["customer_id"])
    cy = pd.Series(y, index=tr_apps["customer_id"])
    row_fold = od["customer_id"].map(cfold).fillna(-1).astype(int).values  # -1 = customer test
    y_row = od["customer_id"].map(cy).fillna(0).astype(int).values
    tr_c, te_c = tr_apps["customer_id"].values, apps.loc[apps["is_test"].eq(1).values, "customer_id"].values
    out = {}
    for k in range(N_FOLDS):
        t = time.time()
        mk = fit_rows(od, (row_fold >= 0) & (row_fold != k), y_row, SEED + k)
        rows_va, rows_te = row_fold == k, row_fold == -1
        f_va = aggregate(od, mk.predict(od.loc[rows_va, FEATS], raw_score=True), rows_va, tr_c[fold == k])
        f_te = aggregate(od, mk.predict(od.loc[rows_te, FEATS], raw_score=True), rows_te, te_c)
        parts = []
        for j in range(N_FOLDS):
            if j == k:
                continue
            mj = fit_rows(od, (row_fold >= 0) & (row_fold != k) & (row_fold != j), y_row, SEED + 10 * k + j)
            rows_j = row_fold == j
            parts.append(aggregate(od, mj.predict(od.loc[rows_j, FEATS], raw_score=True), rows_j, tr_c[fold == j]))
        f_tr = pd.concat(parts).reindex(tr_c[fold != k])
        out[f"tr{k}"], out[f"va{k}"], out[f"te{k}"] = f_tr.values, f_va.values, f_te.values
        print(f"ecm fold {k}: AUC ecm_mean val {roc_auc_score(y[fold == k], f_va['ecm_mean'].fillna(-9)):.4f} "
              f"({(time.time() - t) / 60:.1f} mnt)", flush=True)
    out["cols"] = np.array(f_va.columns)
    del od
    gc.collect()
    return out


ECM = build_ecm(apps, y, fold)

## 7. Model

Matriks latih per fold = 546 kolom basis + 7 kolom `ecm_*` (nama berakhiran `0`, sama dengan pipeline eksperimen).
Parameter hasil Optuna Fase 5 (study `tune_lgb`, 100 trial, set B3) — `RUN_TUNING = False`, jadi di-hardcode:

- **GBDT** (lr final 0,02, batas 15000 iterasi, early stopping 200 di fold validasi): `{'objective': 'binary', 'metric': 'auc', 'learning_rate': 0.02, 'num_leaves': 35, 'min_child_samples': 34, 'feature_fraction': 0.12870618888259533, 'bagging_fraction': 0.9264683082866251, 'bagging_freq': 1, 'lambda_l2': 7.664965525069028, 'max_bin': 255, 'seed': 42, 'deterministic': True, 'force_row_wise': True, 'num_threads': 4, 'verbose': -1, 'max_depth': 4, 'lambda_l1': 0.011885886537829151, 'min_gain_to_split': 0.9382501776308856, 'extra_trees': True}`
- **DART** (lr 0,05, 5000 ronde tetap, drop_rate 0,1, skip_drop 0,5, max_drop 50): parameter pohon sama.

Tiap model dilatih 3 seed (42, 43, 44) × 5 fold; prediksi test = rata-rata 5 model fold per seed.

In [ ]:
PARAMS_LGB = {'objective': 'binary', 'metric': 'auc', 'learning_rate': 0.02, 'num_leaves': 35, 'min_child_samples': 34, 'feature_fraction': 0.12870618888259533, 'bagging_fraction': 0.9264683082866251, 'bagging_freq': 1, 'lambda_l2': 7.664965525069028, 'max_bin': 255, 'seed': 42, 'deterministic': True, 'force_row_wise': True, 'num_threads': 4, 'verbose': -1, 'max_depth': 4, 'lambda_l1': 0.011885886537829151, 'min_gain_to_split': 0.9382501776308856, 'extra_trees': True}
PARAMS_DART = {'objective': 'binary', 'metric': 'auc', 'learning_rate': 0.05, 'num_leaves': 35, 'min_child_samples': 34, 'feature_fraction': 0.12870618888259533, 'bagging_fraction': 0.9264683082866251, 'bagging_freq': 1, 'lambda_l2': 7.664965525069028, 'max_bin': 255, 'seed': 42, 'deterministic': True, 'force_row_wise': True, 'num_threads': 4, 'verbose': -1, 'max_depth': 4, 'lambda_l1': 0.011885886537829151, 'min_gain_to_split': 0.9382501776308856, 'extra_trees': True, 'boosting': 'dart', 'drop_rate': 0.1, 'skip_drop': 0.5, 'max_drop': 50}
SEEDS = [42, 43, 44]
X_tr, X_te = X[m_tr].reset_index(drop=True), X[~m_tr].reset_index(drop=True)
cols_ecm = [f"{c}0" for c in ECM["cols"]]


def fold_mats(k):
    tr, va = fold != k, fold == k
    A = pd.concat([X_tr[tr].reset_index(drop=True), pd.DataFrame(ECM[f"tr{k}"], columns=cols_ecm)], axis=1)
    B = pd.concat([X_tr[va].reset_index(drop=True), pd.DataFrame(ECM[f"va{k}"], columns=cols_ecm)], axis=1)
    C = pd.concat([X_te, pd.DataFrame(ECM[f"te{k}"], columns=cols_ecm)], axis=1)
    return tr, va, A, B, C


def ks_score(y_true, p):
    fpr, tpr, _ = roc_curve(y_true, p)
    return float(np.max(tpr - fpr))


def report(name, oof):
    aucs = [roc_auc_score(y[fold == k], oof[fold == k]) for k in range(N_FOLDS)]
    print(f"{name}: mean-fold AUC {np.mean(aucs):.5f} {np.round(aucs, 4).tolist()} | KS OOF {ks_score(y, oof):.4f}", flush=True)

### 7.1 GBDT tuned (E205)

In [ ]:
OOF_GBDT, PRED_GBDT = {}, {}
for s in SEEDS:
    oof, pred, its = np.zeros(len(y)), np.zeros(len(X_te)), []
    for k in range(N_FOLDS):
        tr, va, A, B, C = fold_mats(k)
        dtr = lgb.Dataset(A, y[tr], params={"max_bin": PARAMS_LGB.get("max_bin", 255)})
        dva = lgb.Dataset(B, y[va], reference=dtr)
        mdl = lgb.train({**PARAMS_LGB, "seed": s}, dtr, 15000, valid_sets=[dva],
                        callbacks=[lgb.early_stopping(200, verbose=False)])
        oof[va] = mdl.predict(B, num_iteration=mdl.best_iteration)
        pred += mdl.predict(C, num_iteration=mdl.best_iteration) / N_FOLDS
        its.append(mdl.best_iteration)
        del A, B, C, dtr, dva, mdl
        gc.collect()
    OOF_GBDT[s], PRED_GBDT[s] = oof, pred
    report(f"GBDT seed {s} (iter {its})", oof)
report("GBDT rata-rata 3 seed", np.mean([OOF_GBDT[s] for s in SEEDS], 0))
print(f"({(time.time() - T0) / 60:.1f} mnt sejak awal)")

### 7.2 DART (E207)
DART tidak memakai early stopping (jumlah ronde tetap 5000); prediksi memakai semua ronde.

In [ ]:
OOF_DART, PRED_DART = {}, {}
for s in SEEDS:
    oof, pred = np.zeros(len(y)), np.zeros(len(X_te))
    for k in range(N_FOLDS):
        tr, va, A, B, C = fold_mats(k)
        dtr = lgb.Dataset(A, y[tr], params={"max_bin": PARAMS_DART.get("max_bin", 255)})
        dva = lgb.Dataset(B, y[va], reference=dtr)
        ev = {}
        mdl = lgb.train({**PARAMS_DART, "seed": s, "drop_seed": s}, dtr, 5000, valid_sets=[dva],
                        callbacks=[lgb.record_evaluation(ev)])
        oof[va] = mdl.predict(B)
        pred += mdl.predict(C) / N_FOLDS
        print(f"  DART seed {s} fold {k}: AUC {roc_auc_score(y[va], oof[va]):.5f} ({(time.time() - T0) / 60:.1f} mnt)", flush=True)
        del A, B, C, dtr, dva, mdl
        gc.collect()
    OOF_DART[s], PRED_DART[s] = oof, pred
    report(f"DART seed {s}", oof)
report("DART rata-rata 3 seed", np.mean([OOF_DART[s] for s in SEEDS], 0))

## 8. Ensemble dan submission

Per model: rank dari rata-rata rank antar seed (urutan seed 42, 43, 44). Blend = rata-rata berbobot 0,5/0,5 dari rank
ternormalisasi; skor akhir = rank blend / 30.000 dibulatkan 6 desimal (KS hanya bergantung pada urutan). File ditulis
dengan urutan `sample_submission` dan akhir baris LF.

In [ ]:
def bag(arrs):
    return rankdata(np.mean([rankdata(a) for a in arrs], 0))


parts = [(bag([OOF_GBDT[s] for s in SEEDS]), bag([PRED_GBDT[s] for s in SEEDS]), 0.5),
         (bag([OOF_DART[s] for s in SEEDS]), bag([PRED_DART[s] for s in SEEDS]), 0.5)]
W = sum(w for _, _, w in parts)
O, T = 0, 0
for o, t, w in parts:
    O, T = O + w / W * o / len(o), T + w / W * t / len(t)
report("Blend GBDT + DART (OOF)", O)
print("KS OOF (ks_2samp):", round(ks_2samp(O[y == 1], O[y == 0]).statistic, 4))
t = (rankdata(T) / len(T)).round(6)
ss = pd.read_csv(DATA_DIR / "sample_submission.csv")
sub = ss[["application_id"]].merge(pd.DataFrame({"application_id": test_ids, "default_probability": t}),
                                   on="application_id", how="left")
assert len(sub) == 30000 and sub.application_id.is_unique and sub.default_probability.notna().all()
assert sub.default_probability.between(0, 1).all()
sub.to_csv(OUT_PATH, index=False, lineterminator="\n")
md5 = hashlib.md5(open(OUT_PATH, "rb").read()).hexdigest()
print("submission:", OUT_PATH, "| MD5", md5, "| identik dengan submission terpilih:", md5 == "afc487f68e4603a38209f2d4a472312b")
print(f"total waktu {(time.time() - T0) / 60:.1f} mnt")

## 9. Interpretasi (SHAP)

Kontribusi fitur model GBDT fold 0 (TreeSHAP bawaan LightGBM, `pred_contrib`). Sel ini dijalankan setelah submission
ditulis dan tidak mengubahnya.

In [ ]:
tr, va, A, B, C = fold_mats(0)
dtr = lgb.Dataset(A, y[tr], params={"max_bin": 255})
mdl = lgb.train({**PARAMS_LGB, "seed": 42}, dtr, 15000, valid_sets=[lgb.Dataset(B, y[va], reference=dtr)],
                callbacks=[lgb.early_stopping(200, verbose=False)])
sv = mdl.predict(B, num_iteration=mdl.best_iteration, pred_contrib=True)[:, :-1]
imp = pd.Series(np.abs(sv).mean(0), index=B.columns).sort_values(ascending=False)
print(imp.head(20).round(4).to_string())

## 10. Kesimpulan

- Sinyal terkuat: riwayat bayar (direpresentasikan sebagai rantai event bulanan), model level order e-commerce
  (ramp belanja menjelang keputusan), skor biro, dan tier bunga lender.
- Residual model tidak lagi bisa dipelajari (uji kapasitas, ±60 uji sinyal, 10.760 fitur otomatis) → model berada dekat
  batas informasi data untuk representasi ini; kenaikan terakhir datang dari representasi yang lebih tepat (rantai bayar)
  dan ensemble GBDT + DART.
- Submission di atas dihasilkan sepenuhnya oleh notebook ini (CPU, deterministik) dengan MD5 yang dicetak di bagian 8.